In [43]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('spark://spark-master:7077')
    .config('spark.executor.instances', 1)  # Is ignored by spark standalone cluster manager
    .config('spark.cores.max', '4')         # Workaround to cap number of executor's instances
    .config('spark.executor.cores', 2)
    .config('spark.executor.memory', '512M')
    .getOrCreate()
)

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark

# small x big

## normal join

In [65]:
emp_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp_df = spark.read.format('csv').schema(emp_schema).option('header', True).load('/home/spark/data/employee/1')

In [38]:
emp_df.printSchema()

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- department_id: integer (nullable = true)



In [39]:
dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept_df = spark.read.format('csv').schema(dept_schema).option('header', True).load('/home/spark/data/department/1')

In [40]:
dept_df.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)



In [46]:
emp_joined_df = emp_df.join(dept_df, how='inner', on='department_id')
emp_joined_df.explain()

== Physical Plan ==
*(5) Project [department_id#113, first_name#106, last_name#107, job_title#108, dob#109, email#110, phone#111, salary#112, department_name#115, description#116, city#117, state#118, country#119]
+- *(5) SortMergeJoin [department_id#113], [department_id#114], Inner
   :- *(2) Sort [department_id#113 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(department_id#113, 200), ENSURE_REQUIREMENTS, [plan_id=553]
   :     +- *(1) Filter isnotnull(department_id#113)
   :        +- FileScan csv [first_name#106,last_name#107,job_title#108,dob#109,email#110,phone#111,salary#112,department_id#113] Batched: false, DataFilters: [isnotnull(department_id#113)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/employee/4], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- *(4) Sort [department_id#114 ASC NULLS FIRST],

## broadcast join

In [47]:
from pyspark.sql.functions import broadcast

emp_joined_2_df = emp_df.join(broadcast(dept_df), how='inner', on='department_id')
emp_joined_2_df.explain()

== Physical Plan ==
*(2) Project [department_id#113, first_name#106, last_name#107, job_title#108, dob#109, email#110, phone#111, salary#112, department_name#115, description#116, city#117, state#118, country#119]
+- *(2) BroadcastHashJoin [department_id#113], [department_id#114], Inner, BuildRight, false
   :- *(2) Filter isnotnull(department_id#113)
   :  +- FileScan csv [first_name#106,last_name#107,job_title#108,dob#109,email#110,phone#111,salary#112,department_id#113] Batched: false, DataFilters: [isnotnull(department_id#113)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/employee/4], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=616]
      +- *(1) Filter isnotnull(department_id#114)
         +- FileScan csv [depar

In [48]:
emp_joined_df.write.mode('overwrite').format('noop').save()

In [49]:
emp_joined_2_df.write.mode('overwrite').format('noop').save()

# big x big

## normal join

In [44]:
sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"

sales_df = spark.read.schema(sales_schema).format('csv').load('/home/spark/data/sales/1')
sales_df.printSchema()

root
 |-- transacted_at: string (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- retailer_id: string (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: string (nullable = true)



In [45]:
cities_schema = "city_id string, city string, state string, state_abv string, country string"

cities_df = spark.read.schema(cities_schema).format('csv').load('/home/spark/data/city/1')
cities_df.printSchema()

root
 |-- city_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- state_abv: string (nullable = true)
 |-- country: string (nullable = true)



In [53]:
sales_joined_df = sales_df.join(cities_df, how='left_outer', on='city_id')
sales_joined_df.explain()

== Physical Plan ==
*(4) Project [city_id#192, transacted_at#187, trx_id#188, retailer_id#189, description#190, amount#191, city#194, state#195, state_abv#196, country#197]
+- *(4) SortMergeJoin [city_id#192], [city_id#193], LeftOuter
   :- *(1) Sort [city_id#192 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(city_id#192, 200), ENSURE_REQUIREMENTS, [plan_id=972]
   :     +- FileScan csv [transacted_at#187,trx_id#188,retailer_id#189,description#190,amount#191,city_id#192] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/sales/1], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit...
   +- *(3) Sort [city_id#193 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(city_id#193, 200), ENSURE_REQUIREMENTS, [plan_id=985]
         +- *(2) Filter isnotnull(city_id#193)
            +- FileScan csv [city_id#193,city#19

In [55]:
sales_joined_df.write.mode('overwrite').format('noop').save()

## bucketing and joining

In [47]:
sales_df.write.format('csv').option('header', True).option('path', '/home/spark/data/sales/2').mode('overwrite').bucketBy(4, 'city_id').saveAsTable('sales_bucket')

In [48]:
cities_df.write.format('csv').option('header', True).option('path', '/home/spark/data/city/2').mode('overwrite').bucketBy(4, 'city_id').saveAsTable('cities_bucket')

In [6]:
spark.sql('show tables').show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|cities_bucket|      false|
|  default| sales_bucket|      false|
+---------+-------------+-----------+



In [49]:
sales_bkt_df = spark.read.table('sales_bucket')
cities_bkt_df = spark.read.table('cities_bucket')

In [51]:
sales_bkt_joined_df = sales_bkt_df.join(cities_bkt_df, how='left_outer', on='city_id')
sales_bkt_joined_df.explain()

== Physical Plan ==
*(3) Project [city_id#214, transacted_at#209, trx_id#210, retailer_id#211, description#212, amount#213, city#216, state#217, state_abv#218, country#219]
+- *(3) SortMergeJoin [city_id#214], [city_id#215], LeftOuter
   :- *(1) Sort [city_id#214 ASC NULLS FIRST], false, 0
   :  +- FileScan csv spark_catalog.default.sales_bucket[transacted_at#209,trx_id#210,retailer_id#211,description#212,amount#213,city_id#214] Batched: false, Bucketed: true, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/sales/2], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit..., SelectedBucketsCount: 4 out of 4
   +- *(2) Sort [city_id#215 ASC NULLS FIRST], false, 0
      +- *(2) Filter isnotnull(city_id#215)
         +- FileScan csv spark_catalog.default.cities_bucket[city_id#215,city#216,state#217,state_abv#218,country#219] Batched: false, Bucketed: tru

In [56]:
sales_bkt_joined_df.write.mode('overwrite').format('noop').save()

In [57]:
spark.stop()